In [2]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.ensemble import EnsembleRetriever
from langchain_community.vectorstores import Chroma

In [3]:
# Load the doc
from langchain_community.document_loaders import PyPDFLoader
documents = PyPDFLoader("Data/rag_paper.pdf").load()

In [4]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\tipto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [5]:
from nltk.tokenize import sent_tokenize

def sentence_based_chunking(text, chunk_size=500, overlap=100):
    sentences = sent_tokenize(text)
    
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)

        # If adding this sentence exceeds chunk size -> finalize chunk
        if current_length + sentence_length > chunk_size:
            chunks.append(" ".join(current_chunk))

            # Handle overlap (take last few sentences)
            overlap_chunk = []
            overlap_length = 0

            for s in reversed(current_chunk):
                if overlap_length + len(s) <= overlap:
                    overlap_chunk.insert(0, s)
                    overlap_length += len(s)
                else:
                    break

            current_chunk = overlap_chunk
            current_length = overlap_length

        # Add new sentence
        current_chunk.append(sentence)
        current_length += sentence_length

    # Add last chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [6]:
from langchain_classic.schema import Document

def chunk_documents(documents, chunk_size=500, overlap=100):
    chunked_docs = []

    for doc in documents:
        text = doc.page_content
        metadata = doc.metadata

        chunks = sentence_based_chunking(
            text,
            chunk_size=chunk_size,
            overlap=overlap
        )

        for i, chunk in enumerate(chunks):
            new_doc = Document(
                page_content=chunk,
                metadata={
                    **metadata,
                    "chunk_id": i
                }
            )
            chunked_docs.append(new_doc)

    return chunked_docs

In [7]:
chunks = chunk_documents(documents)

In [8]:
len(chunks)

183

In [9]:
# a better approach than previous
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_documents_efficiently(documents , chunk_size = 500 , chunk_overlap = 100):
    # initialize the splitter
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        add_start_index = True # tracks where the chunk starts in the original doc
    )
    # split the docs
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [10]:
chunks = chunk_documents_efficiently(
    documents = documents , 
    chunk_size = 500,
    chunk_overlap = 100
)

In [11]:
# split the documents into chunks
# from langchain_text_splitters  import RecursiveCharacterTextSplitter
# splitter = RecursiveCharacterTextSplitter(
#     chunk_size = 500,
#     chunk_overlap = 100
# )
# chunks = splitter.split_documents(documents = documents)

In [ ]:
# from langchain_huggingface import HuggingFaceEmbeddings
# embedding = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-small-en"
# )

# using embedding model from ollama
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(
    model = "nomic-embed-text"
)


In [13]:
# setup the vector store(Dense)
vectorstore = Chroma.from_documents(
    documents = chunks , embedding = embedding
)

In [14]:
# get the dense retriever(semantic retriever)
vector_retriever = vectorstore.as_retriever(
    search_kwagrs = {"k" : 5}
)

In [15]:
query = "Why we need RAG?"
embedd = vector_retriever.invoke(query)

for doc in embedd:
    print(doc.page_content)
    print("-------------------------------------------------------------")

Ł ukasz Kaiser, and Illia Polosukhin. Attention is all you need. In I. Guyon, U. V . Luxburg,
S. Bengio, H. Wallach, R. Fergus, S. Vishwanathan, and R. Garnett, editors,Advances in Neural
Information Processing Systems 30, pages 5998–6008. Curran Associates, Inc., 2017. URL
http://papers.nips.cc/paper/7181-attention-is-all-you-need.pdf .
[59] Ashwin Vijayakumar, Michael Cogswell, Ramprasaath Selvaraju, Qing Sun, Stefan Lee, David
-------------------------------------------------------------
RAG-S The middle ear includes the tympanic cavity and the three ossicles.
what currency
needed in
scotland
BART The currency needed in Scotland is Pound sterling.
RAG-T Pound is the currency needed in Scotland.
RAG-S The currency needed in Scotland is the pound sterling.
Jeopardy
Question
Gener
-ation
Washington
BART ?This state has the largest number of counties in the U.S.
RAG-T It’s the only U.S. state named for a U.S. president
-------------------------------------------------------------
[9] Em

In [16]:
len(embedd)

4

In [17]:
# get the sparse retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 5

In [18]:
embedd = bm25_retriever.invoke(query)
for doc in embedd:
    print(doc.page_content)
    print("-------------------------------------------------------------")

output sequences,|Y| can become large, requiring many forward passes. For more efﬁcient decoding,
we can make a further approximation thatpθ(y|x,zi)≈ 0 wherey was not generated during beam
search fromx,zi. This avoids the need to run additional forward passes once the candidate set Y has
been generated. We refer to this decoding procedure as “Fast Decoding.”
3 Experiments
We experiment with RAG in a wide range of knowledge-intensive tasks. For all experiments, we use
-------------------------------------------------------------
improves results on all other tasks, especially for Open-Domain QA, where it is crucial.
Index hot-swapping An advantage of non-parametric memory models like RAG is that knowledge
can be easily updated at test time. Parametric-only models like T5 or BART need further training to
update their behavior as the world changes. To demonstrate, we build an index using the DrQA [5]
Wikipedia dump from December 2016 and compare outputs from RAG using this index to the ne

In [19]:
# create the hybrid retriever
hybrid_retriever = EnsembleRetriever(
    retrievers = [vector_retriever , bm25_retriever],
    weights = [0.5 , 0.5]
)

In [20]:
# so now we want to re-ranker
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_community.document_compressors.flashrank_rerank import FlashrankRerank

compressor = FlashrankRerank() # we can also use cohere, but flash is fast.

# wrap the hybrid retriever with the reranker
rerank_retriever = ContextualCompressionRetriever(
    base_compressor = compressor,
    base_retriever = hybrid_retriever
)

In [21]:
from langchain_ollama import ChatOllama

# Get the ollama chat model
llm = ChatOllama(
    model = "smollm2:1.7b",
    temperature = 0.1
)

In [22]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system" , "Answer the user's question using ONLY the context provided: \n\n{context}"),
    ("human", "{input}")
])

In [23]:
# now we have multiple chunks of documents from retriever, so we have to merge/stuff
# them into a single context so that we can pass this to the llm

# This chain takes your list of documents, converts them into a single string, 
# and inserts that string into a placeholder (usually called {context}) within prompt.

from langchain_classic.chains.combine_documents import create_stuff_documents_chain

combine_docs_chain = create_stuff_documents_chain(
    llm = llm , prompt = prompt
)

In [24]:
#  now create the final rag chain
from langchain_classic.chains.retrieval import create_retrieval_chain
rag_chain = create_retrieval_chain(
    retriever = hybrid_retriever,
    combine_docs_chain = combine_docs_chain
)

In [25]:
from langchain_core.globals import set_debug
set_debug(False)

In [26]:
response = rag_chain.invoke({
    "input": "Why we need RAG?"
})

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


In [27]:
print(response['answer'])

RAG is needed because it provides a more accurate and efficient way to generate text compared to traditional seq2seq models like BERT or T5. Here are some reasons why:

1. **Memory-based generation**: RAG uses memory from previous outputs to inform the next one, which allows for better contextual understanding of the input question. This leads to more accurate and coherent responses.

2. **Non-parametric memory models**: Unlike seq2seq models that rely on a fixed number of parameters, RAG's memory is learned end-to-end from data. This makes it more flexible and adaptable to different tasks and domains.

3. **Improved performance**: RAG has been shown to outperform traditional seq2seq models in various knowledge-intensive NLP tasks like Open-Domain QA, Knowledge-Intensive Question Answering (KIQA), and Open-Domain Question Answering with Multiple Answers (ODQA).

4. **Flexibility and scalability**: RAG can be easily adapted to different domains and tasks by fine-tuning the model on a sp

## ReRanking with cross-encoder

In [32]:
from langchain_classic.retrievers.document_compressors.cross_encoder_rerank import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# get the cross-encoder model from HF
cross_encoder_model = HuggingFaceCrossEncoder(
    model_name = "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/cross-encoder/ms-marco-MiniLM-L6-v2/c5ee24cb16019beea0893ab7796b1df96625c6b8/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:http

In [37]:
cross_encoder_compressor = CrossEncoderReranker(
    model = cross_encoder_model , top_n = 3
)

# now marge this with normal retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor = cross_encoder_compressor,
    base_retriever = vector_retriever
) 

In [38]:
# define the stuff chain
doc_chain_combined = create_stuff_documents_chain(
    llm = llm , prompt = prompt
)

In [39]:
# create the final rag chain
rag_chain2 = create_retrieval_chain(
    retriever = compression_retriever, 
    combine_docs_chain = doc_chain_combined
)

In [40]:
response2 = rag_chain2.invoke({
    "input": "Why we need RAG?"
})
print(response2['answer'])

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


RAG stands for Recurrent Attention Generative Adversarial Network. It is a type of neural network that combines the strengths of recurrent neural networks (RNNs) and generative adversarial networks (GANs). The primary purpose of RAG is to generate text or images based on user input, while also incorporating attention mechanisms to focus on specific parts of the input data.

RAG was introduced in a 2019 paper titled "Attention Is All You Need" by Lukas Kaiser, Illia Polosukhin, and others. The authors proposed RAG as an alternative to traditional GANs for text generation tasks, such as language translation or image captioning.

The main benefits of using RAG include:

1. Improved text quality: By incorporating attention mechanisms, RAG can focus on specific parts of the input data, resulting in more coherent and context-aware generated text.
2. Enhanced contextual understanding: RAG's ability to incorporate attention allows it to better understand the relationships between different par

In [43]:
# check what the retriever is sending to the llm as context
docs = compression_retriever.invoke("Why we need RAG?")
print(f"Number of docs retrieved: {len(docs)}")
for i, doc in enumerate(docs):
    print(" - " * 20)
    print(f"Doc {i}: {doc.page_content}")

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Number of docs retrieved: 3
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
Doc 0: RAG-S The middle ear includes the tympanic cavity and the three ossicles.
what currency
needed in
scotland
BART The currency needed in Scotland is Pound sterling.
RAG-T Pound is the currency needed in Scotland.
RAG-S The currency needed in Scotland is the pound sterling.
Jeopardy
Question
Gener
-ation
Washington
BART ?This state has the largest number of counties in the U.S.
RAG-T It’s the only U.S. state named for a U.S. president
 -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  -  - 
Doc 1: Ł ukasz Kaiser, and Illia Polosukhin. Attention is all you need. In I. Guyon, U. V . Luxburg,
S. Bengio, H. Wallach, R. Fergus, S. Vishwanathan, and R. Garnett, editors,Advances in Neural
Information Processing Systems 30, pages 5998–6008. Curran Associates, Inc., 2017. URL
http://papers.nips.cc/paper/7181-attention-is-all-you-need.pdf .
[59] Ashwin Vijayakumar, Michael Cogswell, Ramprasaath Selva